In [ ]:
# IndicBART for Nepali Abstractive Summarization
#
# Dependencies (install once):
#   pip install torch pandas numpy scikit-learn transformers \
#               datasets sentencepiece accelerate \
#               sacrebleu bert-score tqdm
#
# Environment:
#   INDICBART_DATA_DIR    folder containing small.csv, medium.csv, large.csv
#                         (default: ./data/)
#   INDICBART_WORK_DIR    output folder (default: ./indicbart_workspace/)
#   INDICBART_BASE_MODEL  HF model id or local path
#                         (default: ai4bharat/IndicBART)
#
# Decoder prefix: <2hi> language token prepended to every target sequence.
# Decoding: beam search with beam size 5, length penalty 2.0, min length 20.

import os
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn
from tqdm.auto import tqdm


# Configuration
CONFIG = {
    "data_dir":   Path(os.environ.get("INDICBART_DATA_DIR", "./data")),
    "work_dir":   Path(os.environ.get("INDICBART_WORK_DIR", "./indicbart_workspace")),
    "base_model": os.environ.get("INDICBART_BASE_MODEL", "ai4bharat/IndicBART"),

    # 80:10:10 split
    "test_size":    0.2,
    "random_state": 42,

    # Tokenization
    "max_input_len":  512,
    "max_target_len": 128,
    "lang_token":     "<2hi>",

    # Common fine-tuning configuration for all pretrained models
    "num_epochs":       3,
    "per_device_batch": 8,
    "grad_accum_steps": 1,
    "learning_rate":    5e-5,
    "fp16":             True,

    # Decoding
    "gen_max_length":     128,
    "gen_num_beams":      5,
    "gen_length_penalty": 2.0,
    "gen_min_length":     20,

    # Inference
    "infer_batch_size":  8,
    "infer_num_workers": 2,
}


# Derived paths and runtime setup
WORK_DIR        = CONFIG["work_dir"]
CKPT_DIR        = WORK_DIR / "indicbart-finetuned"
FINAL_MODEL_DIR = WORK_DIR / "indicbart-final"
TEST_RAW_CSV    = WORK_DIR / "indicbart_test_raw.csv"
PRED_CSV        = WORK_DIR / "indicbart_test_predictions.csv"
SCORES_CSV      = WORK_DIR / "indicbart_final_scores.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("CUDA GPU required.")

WORK_DIR.mkdir(parents=True, exist_ok=True)


# Load data, clean, split 80/10/10
def load_and_split(cfg):
    data_dir = cfg["data_dir"]
    required = ("small.csv", "medium.csv", "large.csv")

    if not data_dir.is_dir():
        raise FileNotFoundError(
            f"Data directory not found: {data_dir.resolve()}. "
            f"Set INDICBART_DATA_DIR or place {', '.join(required)} in ./data/."
        )

    missing = [name for name in required if not (data_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Data directory {data_dir.resolve()} is missing: {', '.join(missing)}."
        )

    shards = [pd.read_csv(data_dir / name) for name in required]
    df = pd.concat(shards, ignore_index=True)

    for col in ("content", "summary"):
        if col not in df.columns:
            raise ValueError(
                f"CSV files must contain a '{col}' column. "
                f"Found columns: {list(df.columns)}"
            )

    df = df.dropna(subset=["content", "summary"]).copy()
    df["content"] = df["content"].astype(str).str.strip()
    df["summary"] = df["summary"].astype(str).str.strip()
    df = df[(df["content"] != "") & (df["summary"] != "")]
    df = df.drop_duplicates(subset=["content"], keep="first").reset_index(drop=True)

    train_df, temp_df = train_test_split(
        df, test_size=cfg["test_size"], random_state=cfg["random_state"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=cfg["random_state"]
    )
    return train_df, val_df, test_df


train_df, val_df, test_df = load_and_split(CONFIG)
test_df[["content", "summary"]].to_csv(TEST_RAW_CSV, index=False, encoding="utf-8")


# Load IndicBART and configure special tokens
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])
model     = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["base_model"])

# Gradient checkpointing reduces activation memory at a small compute cost.
model.gradient_checkpointing_enable()

# The <2hi> language token is used as the decoder start token.
HI_ID = tokenizer.convert_tokens_to_ids(CONFIG["lang_token"])
if HI_ID == tokenizer.unk_token_id:
    raise RuntimeError(
        f"Token {CONFIG['lang_token']} not in vocab. Verify that "
        f"CONFIG['base_model'] is an IndicBART checkpoint."
    )

model.config.decoder_start_token_id = HI_ID
model.config.pad_token_id           = tokenizer.pad_token_id
model.config.eos_token_id           = tokenizer.eos_token_id


# Tokenize: prepend <2hi> to every target sequence
def build_tokenize_fn(tok, hi_id, cfg):
    """Return a batched map function that prepends <2hi> to each target."""
    def tokenize_fn(batch):
        encoded = tok(
            batch["content"],
            max_length=cfg["max_input_len"],
            truncation=True,
            padding=False,
        )
        labels = []
        for summary in batch["summary"]:
            ids = tok(
                summary,
                max_length=cfg["max_target_len"] - 1,
                truncation=True,
                padding=False,
            )["input_ids"]
            labels.append([hi_id] + ids)
        encoded["labels"] = labels
        return encoded
    return tokenize_fn


tokenize_fn = build_tokenize_fn(tokenizer, HI_ID, CONFIG)

tok_train = (
    HFDataset.from_pandas(train_df[["content", "summary"]].reset_index(drop=True))
    .map(tokenize_fn, batched=True, remove_columns=["content", "summary"])
)
tok_val = (
    HFDataset.from_pandas(val_df[["content", "summary"]].reset_index(drop=True))
    .map(tokenize_fn, batched=True, remove_columns=["content", "summary"])
)

assert tok_train[0]["labels"][0] == HI_ID, "Decoder prefix <2hi> was not applied."


# Configure training
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CKPT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,
    per_device_train_batch_size=CONFIG["per_device_batch"],
    per_device_eval_batch_size=CONFIG["per_device_batch"],
    gradient_accumulation_steps=CONFIG["grad_accum_steps"],
    num_train_epochs=CONFIG["num_epochs"],
    learning_rate=CONFIG["learning_rate"],
    fp16=CONFIG["fp16"],
    predict_with_generate=True,
    generation_max_length=CONFIG["gen_max_length"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
)

# transformers >= 4.46 uses `processing_class=`; older versions use `tokenizer=`.
try:
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        data_collator=data_collator,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        data_collator=data_collator,
    )
    trainer.tokenizer = tokenizer


# Train
trainer.train()


# Save the fine-tuned model
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
shutil.make_archive(str(WORK_DIR / "indicbart-finetuned"), "zip", FINAL_MODEL_DIR)


# Batch inference on the held-out test set
def strip_special_tokens(text, tokens=None):
    """Remove residual control tokens from generated text."""
    if tokens is None:
        tokens = ["<2hi>", "<2ne>", "<pad>", "<unk>", "<s>", "</s>"]
    for tok in tokens:
        text = text.replace(tok, "")
    return " ".join(text.split()).strip()


class ContentDataset(Dataset):
    """Minimal wrapper so a DataLoader can batch raw strings."""
    def __init__(self, frame, column="content"):
        self.texts = frame[column].astype(str).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]


def run_inference(cfg, test_frame):
    """Reload the fine-tuned model from disk and generate for every row."""
    infer_model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_MODEL_DIR).to(device).eval()
    infer_tok   = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
    hi_id       = infer_tok.convert_tokens_to_ids(cfg["lang_token"])

    def collate(batch):
        return infer_tok(
            batch,
            max_length=cfg["max_input_len"],
            truncation=True,
            padding=True,
            return_tensors="pt",
        )

    loader = DataLoader(
        ContentDataset(test_frame),
        batch_size=cfg["infer_batch_size"],
        shuffle=False,
        collate_fn=collate,
        num_workers=cfg["infer_num_workers"],
        pin_memory=True,
    )

    gen_kwargs = dict(
        forced_bos_token_id=hi_id,
        max_length=cfg["gen_max_length"],
        num_beams=cfg["gen_num_beams"],
        length_penalty=cfg["gen_length_penalty"],
        early_stopping=True,
        min_length=cfg["gen_min_length"],
    )

    preds = []
    for batch in tqdm(loader, desc="Generating"):
        with torch.no_grad():
            gen = infer_model.generate(
                batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                **gen_kwargs,
            )
        decoded = infer_tok.batch_decode(gen, skip_special_tokens=True)
        preds.extend(strip_special_tokens(t) for t in decoded)
    return preds


test_df_reloaded = pd.read_csv(TEST_RAW_CSV)
predictions = run_inference(CONFIG, test_df_reloaded)

pred_df = pd.DataFrame({
    "source":    test_df_reloaded["content"].astype(str),
    "reference": test_df_reloaded["summary"].astype(str),
    "predicted": predictions,
})
pred_df.to_csv(PRED_CSV, index=False, encoding="utf-8")


# Evaluation
pred_df = pd.read_csv(PRED_CSV)

# Replace empty hypotheses with a single Nepali danda so tokenizer-based
# metrics do not fail on zero-length strings.
empty_mask = pred_df["predicted"].astype(str).str.strip() == ""
n_empty = int(empty_mask.sum())
if n_empty:
    pred_df.loc[empty_mask, "predicted"] = "।"

refs = pred_df["reference"].astype(str).tolist()
hyps = pred_df["predicted"].astype(str).tolist()

# SacreBLEU
bleu_flores = BLEU(tokenize="flores200", max_ngram_order=4).corpus_score(hyps, [refs]).score
bleu_13a    = BLEU(tokenize="13a",      max_ngram_order=4).corpus_score(hyps, [refs]).score


# ROUGE (n-gram F1 and LCS-based F1)
def _ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def rouge_n_f1(hyp_tokens, ref_tokens, n):
    h_ng, r_ng = _ngrams(hyp_tokens, n), _ngrams(ref_tokens, n)
    if not h_ng or not r_ng:
        return 0.0
    overlap = sum((Counter(h_ng) & Counter(r_ng)).values())
    p, r = overlap / len(h_ng), overlap / len(r_ng)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def rouge_l_f1(hyp_tokens, ref_tokens):
    m, n = len(hyp_tokens), len(ref_tokens)
    if m == 0 or n == 0:
        return 0.0
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = (
                dp[i - 1][j - 1] + 1
                if hyp_tokens[i - 1] == ref_tokens[j - 1]
                else max(dp[i - 1][j], dp[i][j - 1])
            )
    lcs = dp[m][n]
    p, r = lcs / m, lcs / n
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


token_pairs = [(h.split(), r.split()) for h, r in zip(hyps, refs)]
r1 = float(np.mean([rouge_n_f1(h, r, 1) for h, r in token_pairs]))
r2 = float(np.mean([rouge_n_f1(h, r, 2) for h, r in token_pairs]))
rL = float(np.mean([rouge_l_f1(h, r)    for h, r in token_pairs]))


# BERTScore using bert-base-multilingual-cased.
P, R, F1 = bert_score_fn(
    hyps, refs,
    model_type="bert-base-multilingual-cased",
    verbose=False,
    device=device.type,
)
bs_p, bs_r, bs_f1 = P.mean().item(), R.mean().item(), F1.mean().item()


# Jaccard similarity
def _jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


def jaccard_word(a, b):
    return _jaccard(set(str(a).split()), set(str(b).split()))


def jaccard_char_bigram(a, b):
    a, b = str(a), str(b)
    grams_a = {a[i:i + 2] for i in range(max(len(a) - 1, 0))}
    grams_b = {b[i:i + 2] for i in range(max(len(b) - 1, 0))}
    return _jaccard(grams_a, grams_b)


jac_w = float(np.mean([jaccard_word(h, r)        for h, r in zip(hyps, refs)]))
jac_c = float(np.mean([jaccard_char_bigram(h, r) for h, r in zip(hyps, refs)]))

summary = {
    "model":               "IndicBART (3 epochs)",
    "sacrebleu_flores200": round(bleu_flores, 4),
    "sacrebleu_13a":       round(bleu_13a,    4),
    "rouge1":              round(r1, 4),
    "rouge2":              round(r2, 4),
    "rougeL":              round(rL, 4),
    "bertscore_p":         round(bs_p,  4),
    "bertscore_r":         round(bs_r,  4),
    "bertscore_f1":        round(bs_f1, 4),
    "jaccard_word":        round(jac_w, 4),
    "jaccard_char":        round(jac_c, 4),
    "n_samples":           len(refs),
    "n_empty":             n_empty,
    "avg_pred_chars":      round(float(np.mean([len(h) for h in hyps])), 1),
    "avg_ref_chars":       round(float(np.mean([len(r) for r in refs])), 1),
}

pd.Series(summary).to_csv(SCORES_CSV)